# Librerias

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import visualization
from scipy.stats import norm

import scipy.stats as si
import sympy as sy
from sympy.stats import Normal, cdf

from scipy.stats import multivariate_normal as mvn
from scipy.optimize import minimize

# Funciones

In [2]:
# EUROPEAN OPTIONS
def EurCall(So, K, r, v, T, n):
    dt = T/n

    u = np.exp(v*np.sqrt(dt))
    d = 1/u
    p = (np.exp(r*dt)-d)/(u-d)

    # Stock.
    stock = np.zeros((n+1, n+1))
    stock[0, 0] = So

    # Create tree So
    for i in range(1, n+1):
        stock[i, 0] = stock[i-1, 0]*u
        for j in range(1, i+1):
            stock[i, j] = stock[i-1, j-1]*d

    # option Value
    option = np.zeros((n+1, n+1))

    # Last nodes
    for k in range(n+1):
        option[n, k] = max(stock[n, k]-K, 0)

    # Middle nodes
    for i in range(n-1, -1, -1):
        for j in range(i+1):
            option[i, j] = np.exp(-r*dt)*(p*option[i+1, j]+(1-p)*option[i+1, j+1])

    return option[0, 0]


def EurPut(So, K, r, v, T, n):
    dt = T/n

    u = np.exp(v*np.sqrt(dt))
    d = 1/u
    p = (np.exp(r*dt)-d)/(u-d)

    # Stock.
    stock = np.zeros((n+1, n+1))
    stock[0, 0] = So

    # Create tree So
    for i in range(1, n+1):
        stock[i, 0] = stock[i-1, 0]*u
        for j in range(1, i+1):
            stock[i, j] = stock[i-1, j-1]*d

    # option Value
    option = np.zeros((n+1, n+1))

    # Last nodes
    for k in range(n+1):
        option[n, k] = max(K-stock[n, k], 0)

    # Middle nodes
    for i in range(n-1, -1, -1):
        for j in range(i+1):
            option[i, j] = np.exp(-r*dt)*(p*option[i+1, j]+(1-p)*option[i+1, j+1])

    return option[0, 0]


# AMERICAN OPTIONS
def AmeCall(So, K, r, v, T, n):
    dt = T/n

    u = np.exp(v*np.sqrt(dt))
    d = 1/u
    p = (np.exp(r*dt)-d)/(u-d)

    # Stock.
    stock = np.zeros((n+1, n+1))
    stock[0, 0] = So

    # Create tree So
    for i in range(1, n+1):
        stock[i, 0] = stock[i-1, 0]*u
        for j in range(1, i+1):
            stock[i, j] = stock[i-1, j-1]*d

    # option Value
    option = np.zeros((n+1, n+1))

    # Last nodes
    for k in range(n+1):
        option[n, k] = max(stock[n, k]-K, 0)

    # Middle nodes
    for i in range(n-1, -1, -1):
        for j in range(i+1):
            F1 = np.exp(-r*dt)*(p*option[i+1, j]+(1-p)*option[i+1, j+1])
            F2 = max(stock[i, j]-K, 0)

            option[i, j] = max(F1, F2)

    return option[0, 0]


def AmePut(So, K, r, v, T, n):
    dt = T/n

    u = np.exp(v*np.sqrt(dt))
    d = 1/u
    p = (np.exp(r*dt)-d)/(u-d)

    # Stock.
    stock = np.zeros((n+1, n+1))
    stock[0, 0] = So

    # Create tree So
    for i in range(1, n+1):
        stock[i, 0] = stock[i-1, 0]*u
        for j in range(1, i+1):
            stock[i, j] = stock[i-1, j-1]*d

    # option Value
    option = np.zeros((n+1, n+1))

    # Last nodes
    for k in range(n+1):
        option[n, k] = max(K-stock[n, k], 0)

    # Middle nodes
    for i in range(n-1, -1, -1):
        for j in range(i+1):
            F1 = np.exp(-r*dt)*(p*option[i+1, j]+(1-p)*option[i+1, j+1])
            F2 = max(K-stock[i, j], 0)

            option[i, j] = max(F1, F2)

    return option[0, 0]


# === FINANCIAL OPTIONS === #
def e_call(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = (np.log(S / K) + (r - 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    call = (S * si.norm.cdf(d1, 0.0, 1.0) - K *
            np.exp(-r * T) * si.norm.cdf(d2, 0.0, 1.0))
    return call


def e_put(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = (np.log(S / K) + (r - 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    put = (K * np.exp(-r * T) * si.norm.cdf(-d2, 0.0, 1.0) -
           S * si.norm.cdf(-d1, 0.0, 1.0))
    return put


# === OPTIMAL OPTIONS PRICE === #
def SOpt_call(S, X1, X2, T1, T2, r, sigma):
    tau = T2-T1
    bounds = [[0, None]]
    def apply_constraint1(S): return e_call(S, X2, tau, r, sigma)-X1
    my_constraints = ({'type': 'eq', "fun": apply_constraint1})
    a = minimize(e_call, S, bounds=bounds, constraints=my_constraints,
                 method='SLSQP', args=(X2, tau, r, sigma))
    SOpt = a.x[0]
    return SOpt


def SOpt_put(S, X1, X2, T1, T2, r, sigma):
    tau = T2-T1
    bounds = [[0, None]]
    def apply_constraint1(S): return e_put(S, X2, tau, r, sigma)-X1
    my_constraints = ({'type': 'eq', "fun": apply_constraint1})
    a = minimize(e_put, S, bounds=bounds, constraints=my_constraints,
                 method='SLSQP', args=(X2, tau, r, sigma))
    SOpt = a.x[0]
    return SOpt


# === COMPOUND OPTIONS PRICE === #
def Call_call(S, X1, X2, T1, T2, t, r, sigma):
    SOpt = SOpt_call(S, X1, X2, T1, T2, r, sigma)
    D1 = (np.log(S / SOpt) + (r + 0.5 * sigma ** 2)
          * (T1-t)) / (sigma * np.sqrt(T1-t))
    D2 = D1-sigma*np.sqrt(T1-t)

    E1 = (np.log(S / X2) + (r + 0.5 * sigma ** 2)
          * (T2-t)) / (sigma * np.sqrt(T2-t))
    E2 = E1-sigma*np.sqrt(T2-t)

    Corr = np.sqrt(T1/T2)
    dist = mvn(mean=np.array([0, 0]), cov=np.array([[1, Corr], [Corr, 1]]))
    N2D1E1 = dist.cdf(np.array([D1, E1]))
    N2D2E2 = dist.cdf(np.array([D2, E2]))
    ND2 = si.norm.cdf(D2, 0.0, 1.0)

    Call_call = S*N2D1E1 - X2 * \
        np.exp(-r*(T2-t))*N2D2E2 - X1*np.exp(-r*(T1-t))*ND2
    return Call_call


def Call_put(S, X1, X2, T1, T2, t, r, sigma):
    SOpt = SOpt_put(S, X1, X2, T1, T2, r, sigma)
    D1 = (np.log(S / SOpt) + (r + 0.5 * sigma ** 2)
          * (T1-t)) / (sigma * np.sqrt(T1-t))
    D2 = D1-sigma*np.sqrt(T1-t)

    E1 = (np.log(S / X2) + (r + 0.5 * sigma ** 2)
          * (T2-t)) / (sigma * np.sqrt(T2-t))
    E2 = E1-sigma*np.sqrt(T2-t)

    Corr = np.sqrt(T1/T2)
    dist = mvn(mean=np.array([0, 0]), cov=np.array([[1, Corr], [Corr, 1]]))
    N2_D1_E1 = dist.cdf(np.array([-D1, -E1]))
    N2_D2_E2 = dist.cdf(np.array([-D2, -E2]))
    N_D2 = si.norm.cdf(-D2, 0.0, 1.0)

    Call_put = -S*N2_D1_E1 + X2 * \
        np.exp(-r*(T2-t))*N2_D2_E2 - X1*np.exp(-r*(T1-t))*N_D2
    return Call_put


def Put_call(S, X1, X2, T1, T2, t, r, sigma):
    SOpt = SOpt_call(S, X1, X2, T1, T2, r, sigma)
    D1 = (np.log(S / SOpt) + (r + 0.5 * sigma ** 2)
          * (T1-t)) / (sigma * np.sqrt(T1-t))
    D2 = D1-sigma*np.sqrt(T1-t)

    E1 = (np.log(S / X2) + (r + 0.5 * sigma ** 2)
          * (T2-t)) / (sigma * np.sqrt(T2-t))
    E2 = E1-sigma*np.sqrt(T2-t)

    Corr = np.sqrt(T1/T2)
    dist = mvn(mean=np.array([0, 0]), cov=np.array([[1, -Corr], [-Corr, 1]]))
    N2_D1E1 = dist.cdf(np.array([-D1, E1]))
    N2_D2E2 = dist.cdf(np.array([-D2, E2]))
    N_D2 = si.norm.cdf(-D2, 0.0, 1.0)

    Put_call = -S*N2_D1E1 + X2 * \
        np.exp(-r*(T2-t))*N2_D2E2 + X1*np.exp(-r*(T1-t))*N_D2
    return Put_call


def Put_put(S, X1, X2, T1, T2, t, r, sigma):
    SOpt = SOpt_put(S, X1, X2, T1, T2, r, sigma)
    D1 = (np.log(S / SOpt) + (r + 0.5 * sigma ** 2)
          * (T1-t)) / (sigma * np.sqrt(T1-t))
    D2 = D1-sigma*np.sqrt(T1-t)

    E1 = (np.log(S / X2) + (r + 0.5 * sigma ** 2)
          * (T2-t)) / (sigma * np.sqrt(T2-t))
    E2 = E1-sigma*np.sqrt(T2-t)

    Corr = np.sqrt(T1/T2)
    dist = mvn(mean=np.array([0, 0]), cov=np.array([[1, -Corr], [-Corr, 1]]))
    N2D1_E1 = dist.cdf(np.array([D1, -E1]))
    N2D2_E2 = dist.cdf(np.array([D2, -E2]))
    ND2 = si.norm.cdf(D2, 0.0, 1.0)

    Put_put = S*N2D1_E1 - X2 * \
        np.exp(-r*(T2-t))*N2D2_E2 + X1*np.exp(-r*(T1-t))*ND2
    return Put_put

# Ejercicio 6

In [3]:
S = 500
K = 200
T = 2
sigma = 0.3
r = 0.1

# a)
eur_call_price = e_call(S, K, T, r, sigma)

# b)
eur_put_price = e_put(S, K, T, r, sigma)

# c)
ame_call_price = AmeCall(S, K, r, sigma, T, 20)

# d)
ame_put_price = AmePut(S, K, r, sigma, T, 20)

# Results
print(f'a) European call price: ${eur_call_price:,.4f}')
print(f'b) European put price:  ${eur_put_price:,.4f}')
print(f'c) American call price: ${ame_call_price:,.4f}')
print(f'd) American put price:  ${ame_put_price:,.4f}')

a) European call price: $336.4117
b) European put price:  $0.1579
c) American call price: $336.3990
d) American put price:  $0.1587


# Ejercicio 7

In [4]:
S = 500
K = 500
T = 2
sigma = 0.3
r = 0.1

# a)
eur_call_price = e_call(S, K, T, r, sigma)

# b)
eur_put_price = e_put(S, K, T, r, sigma)

# c)
ame_call_price = AmeCall(S, K, r, sigma, T, 20)

# d)
ame_put_price = AmePut(S, K, r, sigma, T, 20)

# Results
print(f'a) European call price: ${eur_call_price:,.4f}')
print(f'b) European put price:  ${eur_put_price:,.4f}')
print(f'c) American call price: ${ame_call_price:,.4f}')
print(f'd) American put price:  ${ame_put_price:,.4f}')

a) European call price: $129.8774
b) European put price:  $39.2428
c) American call price: $128.8481
d) American put price:  $50.5869


# Ejercicio 8

In [5]:
S = 500
K = 700
T = 2
sigma = 0.3
r = 0.1

# a)
eur_call_price = e_call(S, K, T, r, sigma)

# b)
eur_put_price = e_put(S, K, T, r, sigma)

# c)
ame_call_price = AmeCall(S, K, r, sigma, T, 20)

# d)
ame_put_price = AmePut(S, K, r, sigma, T, 20)

# Results
print(f'a) European call price: ${eur_call_price:,.4f}')
print(f'b) European put price:  ${eur_put_price:,.4f}')
print(f'c) American call price: ${ame_call_price:,.4f}')
print(f'd) American put price:  ${ame_put_price:,.4f}')

a) European call price: $58.1293
b) European put price:  $131.2408
c) American call price: $57.8533
d) American put price:  $200.0000


# Ejercicio 9

In [6]:
S = 500
X1 = 100
X2 = 700
T1 = 0.5
T2 = 2
sigma = 0.3
r = 0.1
t = 0

call_call_price = Call_call(S,X1,X2,T1,T2,t,r,sigma)
call_put_price = Call_put(S,X1,X2,T1,T2,t,r,sigma)
put_call_price = Put_call(S,X1,X2,T1,T2,t,r,sigma)
put_put_price = Put_put(S,X1,X2,T1,T2,t,r,sigma)

print(f'a) Call-call price ${call_call_price:.4f}')
print(f'b) Call-put price  ${call_put_price:.4f}')
print(f'c) Put-call price  ${put_call_price:.4f}')
print(f'd) Put-put price   ${put_put_price:.4f}')

a) Call-call price $9.7587
b) Call-put price  $44.8139
c) Put-call price  $46.7523
d) Put-put price   $8.6960


# Ejercicio 10

In [7]:
S = 500
X1 = 100
X2 = 300
T1 = 0.5
T2 = 2
sigma = 0.3
r = 0.1
t = 0

call_call_price = Call_call(S,X1,X2,T1,T2,t,r,sigma)
call_put_price = Call_put(S,X1,X2,T1,T2,t,r,sigma)
put_call_price = Put_call(S,X1,X2,T1,T2,t,r,sigma)
put_put_price = Put_put(S,X1,X2,T1,T2,t,r,sigma)

print(f'a) Call-call price ${call_call_price:.4f}')
print(f'b) Call-put price  ${call_put_price:.4f}')
print(f'c) Put-call price  ${put_call_price:.4f}')
print(f'd) Put-put price   ${put_put_price:.4f}')

a) Call-call price $162.6824
b) Call-put price  $0.0000
c) Put-call price  $0.5866
d) Put-put price   $92.2850


/var/folders/tr/mb4tgd2j63j6tf50s0kl_p680000gn/T/ipykernel_40949/1182349210.py:147: RuntimeWarning: divide by zero encountered in log
  d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
/var/folders/tr/mb4tgd2j63j6tf50s0kl_p680000gn/T/ipykernel_40949/1182349210.py:148: RuntimeWarning: divide by zero encountered in log
  d2 = (np.log(S / K) + (r - 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))


# Ejercicio 11

In [8]:
S = 500
X1 = 50
X2 = 700
T1 = 0.5
T2 = 2
sigma = 0.3
r = 0.1
t = 0

call_call_price = Call_call(S,X1,X2,T1,T2,t,r,sigma)
call_put_price = Call_put(S,X1,X2,T1,T2,t,r,sigma)
put_call_price = Put_call(S,X1,X2,T1,T2,t,r,sigma)
put_put_price = Put_put(S,X1,X2,T1,T2,t,r,sigma)

print(f'a) Call-call price ${call_call_price:.4f}')
print(f'b) Call-put price  ${call_put_price:.4f}')
print(f'c) Put-call price  ${put_call_price:.4f}')
print(f'd) Put-put price   ${put_put_price:.4f}')

a) Call-call price $23.8618
b) Call-put price  $84.4665
c) Put-call price  $13.2941
d) Put-put price   $0.7872


# Ejercicio 12

In [9]:
S = 500
X1 = 50
X2 = 300
T1 = 0.5
T2 = 2
sigma = 0.3
r = 0.1
t = 0

call_call_price = Call_call(S,X1,X2,T1,T2,t,r,sigma)
call_put_price = Call_put(S,X1,X2,T1,T2,t,r,sigma)
put_call_price = Put_call(S,X1,X2,T1,T2,t,r,sigma)
put_put_price = Put_put(S,X1,X2,T1,T2,t,r,sigma)

print(f'a) Call-call price ${call_call_price:.4f}')
print(f'b) Call-put price  ${call_put_price:.4f}')
print(f'c) Put-call price  ${put_call_price:.4f}')
print(f'd) Put-put price   ${put_put_price:.4f}')

a) Call-call price $209.6749
b) Call-put price  $0.0006
c) Put-call price  $0.0177
d) Put-put price   $44.7241


/var/folders/tr/mb4tgd2j63j6tf50s0kl_p680000gn/T/ipykernel_40949/1182349210.py:147: RuntimeWarning: divide by zero encountered in log
  d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
/var/folders/tr/mb4tgd2j63j6tf50s0kl_p680000gn/T/ipykernel_40949/1182349210.py:148: RuntimeWarning: divide by zero encountered in log
  d2 = (np.log(S / K) + (r - 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
